# ISIC 2024 - Data Preprocessing Pipeline

This notebook builds a preprocessing pipeline for the ISIC 2024 dataset.

## Objectives
- Clean and prepare tabular data
- Handle missing values
- Encode categorical features
- Select usable features (avoid leakage)
- Prepare data for model training

## Key Considerations
- Severe class imbalance
- High missing values in some features
- Train/Test feature mismatch
- Patient-level data already handled in splitting

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

In [2]:
train_df = pd.read_csv("../data/splits/train_split.csv", low_memory=False)
val_df = pd.read_csv("../data/splits/val_split.csv", low_memory=False)
test_df = pd.read_csv("../data/splits/test_split.csv", low_memory=False)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (320848, 55)
Val: (40105, 55)
Test: (40106, 55)


### Feature Selection (Initial)

Removed:
- Diagnostic columns (`iddx_*`, `mel_*`) -> high missing / leakage risk
- Model-derived feature (`tbp_lv_dnn_lesion_confidence`) -> leakage
- Identifiers (`isic_id`, `patient_id`) -> not meaningful for learning

Only features available at inference time are kept.

In [3]:
DROP_COLUMNS = [
    # target leakage / not in test
    "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
    "mel_mitotic_index", "mel_thick_mm",
    "tbp_lv_dnn_lesion_confidence",

    # identifiers (not useful for model)
    "isic_id", "patient_id", "lesion_id"
]

train_df = train_df.drop(columns=[col for col in DROP_COLUMNS if col in train_df.columns])
val_df = val_df.drop(columns=[col for col in DROP_COLUMNS if col in val_df.columns])
test_df = test_df.drop(columns=[col for col in DROP_COLUMNS if col in test_df.columns])

In [4]:
y_train = train_df["target"]
y_val = val_df["target"]

train_df = train_df.drop(columns=["target"])
val_df = val_df.drop(columns=["target"])

### Feature Types

- Categorical features will be encoded
- Numerical features will be imputed and scaled later

In [5]:
categorical_cols = train_df.select_dtypes(include=["object", "string"]).columns.tolist()
numerical_cols = train_df.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)

Categorical: ['sex', 'anatom_site_general', 'image_type', 'tbp_tile_type', 'tbp_lv_location', 'tbp_lv_location_simple', 'attribution', 'copyright_license']
Numerical: ['age_approx', 'clin_size_long_diam_mm', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B', 'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext', 'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio', 'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL', 'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence', 'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt', 'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z']


### Missing Value Strategy

- Numerical → filled with **median** (robust to outliers)
- Categorical → filled with `"unknown"`

This ensures no information leakage from validation/test sets.

In [6]:
# Numerical -> fill with median
for col in numerical_cols:
    median_val = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_val)
    val_df[col] = val_df[col].fillna(median_val)
    test_df[col] = test_df[col].fillna(median_val)

# Categorical → fill with "unknown"
for col in categorical_cols:
    train_df[col] = train_df[col].fillna("unknown")
    val_df[col] = val_df[col].fillna("unknown")
    test_df[col] = test_df[col].fillna("unknown")

### Encoding Strategy

- Label Encoding is used for simplicity
- Encoders are fitted on training data only
- Same mapping is applied to validation and test sets

In [7]:
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    
    train_df[col] = le.fit_transform(train_df[col])
    
    # handle unseen labels
    val_df[col] = val_df[col].map(lambda x: x if x in le.classes_ else "unknown")
    test_df[col] = test_df[col].map(lambda x: x if x in le.classes_ else "unknown")
    
    le.classes_ = np.append(le.classes_, "unknown")
    
    val_df[col] = le.transform(val_df[col])
    test_df[col] = le.transform(test_df[col])
    
    encoders[col] = le

### Scaling

- StandardScaler is applied to numerical features
- Fit only on training data to prevent leakage

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])
val_df[numerical_cols] = scaler.transform(val_df[numerical_cols])
test_df[numerical_cols] = scaler.transform(test_df[numerical_cols])

In [9]:
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (320848, 42)
Val shape: (40105, 42)
Test shape: (40106, 43)


In [10]:
train_df["target"] = y_train
val_df["target"] = y_val

train_df.to_csv("../data/processed/train_processed.csv", index=False)
val_df.to_csv("../data/processed/val_processed.csv", index=False)
test_df.to_csv("../data/processed/test_processed.csv", index=False)

print("Preprocessed data saved!")

Preprocessed data saved!


## Summary

- Removed unusable and leakage-prone features
- Handled missing values
- Encoded categorical variables
- Scaled numerical features

## Ready for:
- PyTorch Dataset / DataLoader
- Model training